## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from torch.utils.data import DataLoader

load_dotenv()
DATA_ROOT = Path(os.getenv("DATA_ROOT"))
NIH_DIR = DATA_ROOT / "NIH_Chest_X-Rays"
DEVICE = "cuda"
_ = torch.manual_seed(0)

## 2. Train-Only Class Statistics

In [2]:
from src.data import NIH_DISEASES, load_dev_labels
from src.losses import compute_pos_weight, compute_prevalence
from src.paths import SPLIT_CSV

df = load_dev_labels(NIH_DIR, SPLIT_CSV)
train_df = df[df["split"] == "train"]

prevalence = compute_prevalence(train_df)
w_ratio = compute_pos_weight(train_df, mode="ratio")
w_sqrt = compute_pos_weight(train_df, mode="sqrt")

stats = pd.DataFrame({
    "n_pos": train_df[NIH_DISEASES].sum().to_numpy(),
    "prevalence": prevalence.double().numpy().round(4),
    "w_ratio": w_ratio.double().numpy().round(1),
    "w_sqrt": w_sqrt.double().numpy().round(1),
    "pos_per_batch64": (64 * prevalence).double().numpy().round(2),
}, index=NIH_DISEASES).sort_values("prevalence")

baseline = -(prevalence * prevalence.log() + (1 - prevalence) * (1 - prevalence).log()).mean()
print(f"Know-nothing baseline (mean base-rate entropy): {baseline:.4f}")
stats

Know-nothing baseline (mean base-rate entropy): 0.1643


,n_pos,prevalence,w_ratio,w_sqrt,pos_per_batch64
Hernia,119,0.0016,637.5,25.2,0.10
Pneumonia,763,0.0100,98.6,9.9,0.64
Fibrosis,1098,0.0145,68.2,8.3,0.92
Emphysema,1249,0.0164,59.8,7.7,1.05
Edema,1250,0.0165,59.8,7.7,1.05
Cardiomegaly,1491,0.0196,50.0,7.1,1.26
Pleural_Thickening,1995,0.0263,37.1,6.1,1.68
Pneumothorax,2340,0.0308,31.5,5.6,1.97
Consolidation,2543,0.0335,28.9,5.4,2.14
Mass,3620,0.0476,20.0,4.5,3.05


## 3. Guard Sanity Check

In [3]:
try:
    compute_prevalence(df)
except ValueError as e:
    print("Refused as expected:", e)

Refused as expected: Class statistics must come from train rows only, got splits: ['train', 'val']


## 4. Where Weighting Moves the Optimal Prediction

In [4]:
pi = 0.01
p = torch.linspace(0.001, 0.999, 9999)

rows = []
for name, w in [("none", 1.0), ("sqrt", ((1 - pi) / pi) ** 0.5), ("ratio", (1 - pi) / pi)]:
    loss = -(w * pi * torch.log(p) + (1 - pi) * torch.log(1 - p))
    rows.append({
        "mode": name,
        "w": round(w, 2),
        "argmin_p (grid)": round(p[loss.argmin()].item(), 4),
        "p* (formula)": round(w * pi / (w * pi + 1 - pi), 4),
        "logit shift = log w": round(float(np.log(w)), 3),
    })
pd.DataFrame(rows)

,mode,w,argmin_p (grid),p* (formula),logit shift = log w
0,none,1.00,0.0100,0.0100,0.000
1,sqrt,9.95,0.0913,0.0913,2.298
2,ratio,99.00,0.5000,0.5000,4.595


## 5. Init-Loss Check for Each Config

In [5]:
from src.dataset import NIHDataset
from src.data import build_image_paths
from src.losses import build_loss, prior_logits
from src.model import build_model, init_head_bias
from src.transforms import build_transforms

train_ds = NIHDataset(train_df, build_image_paths(NIH_DIR), build_transforms(train=False))
torch.manual_seed(0)
x, y = next(iter(DataLoader(train_ds, batch_size=32, shuffle=True)))
x, y = x.to(DEVICE), y.to(DEVICE)

rows = []
for mode in ["none", "sqrt", "ratio"]:
    pos_weight = compute_pos_weight(train_df, mode=mode)
    bias = prior_logits(prevalence, pos_weight)

    torch.manual_seed(0)
    model = build_model().to(DEVICE).eval()
    init_head_bias(model, bias)
    loss_fn = build_loss(pos_weight).to(DEVICE)

    w = torch.ones_like(prevalence) if pos_weight is None else pos_weight
    p0 = torch.sigmoid(bias)
    expected = -(w * prevalence * p0.log() + (1 - prevalence) * (1 - p0).log()).mean()

    with torch.no_grad():
        measured = loss_fn(model(x), y)

    rows.append({"mode": mode, "expected": round(expected.item(), 4), "measured (batch)": round(measured.item(), 4)})

pd.DataFrame(rows)

,mode,expected,measured (batch)
0,none,0.1643,0.1759
1,sqrt,0.4822,0.4926
2,ratio,1.3241,1.3217


## Loss decisions
- Loss: `BCEWithLogitsLoss` (14 independent binary problems, shared backbone).
- Class statistics (prevalence, pos_weight) computed from **train rows only**; `_require_train_only` raises on val/test.
- Know-nothing baseline: mean base-rate entropy = 0.164 (unweighted). Training loss must go below this to show real learning.
- Head bias initialized to the prior: b = log(π/(1−π)) + log w. Starts training at the base-rate solution instead of 50% per class, avoiding large early gradients into the pretrained backbone. (Full-ratio weights give b = 0.)
- `pos_weight` shifts optimal logits by +log w, so probabilities are inflated (π=1%: none->0.01, sqrt->0.09, ratio->0.5). Any calibration analysis of a weighted model needs prior correction (subtract log w).
- Hernia w_ratio = 637 with ~0.1 positives per batch of 64 -> high-variance gradients; sqrt (w=25) is the compromise.
- **Not decided here:** `none` vs `sqrt` (optionally `ratio`) is a **val experiment in milestone 6**, selected by macro AUROC over 13 classes (excluding Hernia), with calibration as a secondary check; ties go to `none`.